# Introduction to Prompt Engineering

When you use ChatGPT, Claude, or Gemini, you communicate through **natural language**. The text you send — instructions, questions, examples, constraints — is called a **prompt**. **Prompt engineering** is the discipline of designing those inputs so a large language model (LLM) produces useful, accurate, and controllable outputs.

Unlike traditional programming, where you write explicit rules (`if x > 0 then ...`), you guide an LLM by describing *what you want* in language. Small changes in wording, structure, or examples can dramatically change the result. Prompt engineering is therefore both a creative and an empirical skill: you form a hypothesis about what to write, call the model, observe the output, and refine.

This notebook introduces the foundations: what prompt engineering is, how it differs from fine-tuning and RAG, why prompts affect quality and cost, the lifecycle of a production prompt, and the mental model you need before diving into techniques like few-shot learning, chain-of-thought, and structured output in later notebooks.

You should have completed **02. GenAI & LLM Fundamentals** (especially tokens and inference parameters) and **03. LLM API Integration** (calling provider APIs) before this section.

---

## 1. What Is Prompt Engineering?

**Prompt engineering** is the practice of crafting, testing, and refining the text (and message structure) sent to an LLM to achieve a desired behavior. It includes:

- **Instructions** — what task the model should perform
- **Context** — background information the model needs
- **Examples** — demonstrations of input/output pairs (few-shot prompting)
- **Constraints** — format, length, tone, and things to avoid
- **Roles** — system-level behavior (tutor, analyst, strict JSON extractor)

The model weights are fixed at inference time. You are not retraining the network; you are **conditioning** generation through the input. Mathematically, an autoregressive LLM computes the probability of the next token given all prior tokens in the context:

$$P(x_t \mid x_1, x_2, \ldots, x_{t-1})$$

Your prompt becomes the prefix $x_1, \ldots, x_{t-1}$ that shapes every token the model generates afterward. Better prefixes steer the distribution toward better completions.

### 1.1 What Prompt Engineering Is Not

Clarifying boundaries prevents confusion as you progress through the curriculum.

**It is not magic wording.** There is no universal "perfect phrase" that fixes every task. Prompt engineering works because models learned patterns from training data — clear instructions resemble helpful text they have seen before.

**It is not a substitute for correct architecture.** If an application needs live data, a prompt alone cannot invent facts. Retrieval, tools, or databases may be required (covered in RAG and agents later).

**It is not only for chatbots.** Prompts appear in classification pipelines, data extraction, code generation, summarization, evaluation rubrics, and automated testing of other prompts.

**It is not permanent.** Models are updated, and prompts that work today may degrade tomorrow. Production systems version and regression-test prompts like any other code.

### 1.2 Prompt Engineering vs Fine-Tuning vs RAG vs Agents

Modern LLM applications combine several techniques. Prompt engineering is usually the **first** and **cheapest** lever.

| Approach | What changes | Typical cost | When to use |
|----------|--------------|--------------|-------------|
| **Prompt engineering** | Input text only | API call cost | Default starting point; fast iteration |
| **Fine-tuning** | Model weights on your data | Training compute + hosting | Stable style/format; domain-specific behavior at scale |
| **RAG** | Retrieves documents into the prompt | Retrieval infra + tokens | Answers must cite private or fresh knowledge |
| **Agents** | Model + tools + multi-step loops | Multiple API calls | Tasks requiring actions (search, DB, APIs) |

These approaches stack. A production assistant might use:

1. A **system prompt** defining role and safety rules (prompt engineering)
2. **Retrieved chunks** inserted into context (RAG)
3. **Tool calls** to fetch live data (agents)
4. A **fine-tuned** model for a specialized tone or format (optional)

For learning, master **prompting first** because every other pattern still sends text to the model. RAG adds context to the prompt; agents chain multiple prompts and tool results.

### 1.3 Why Prompts Matter: Quality, Cost, and Latency

**Quality** — Ambiguous prompts produce ambiguous answers. A prompt that defines the audience, format, and success criteria reduces hallucination-style guessing and improves consistency across runs.

**Cost** — Providers bill by the **token**. Longer prompts (system instructions, few-shot examples, retrieved documents) increase input tokens on every request. Concise, well-structured prompts save money at scale.

**Latency** — More tokens in the context window mean more computation before the first output token, especially for long multi-turn histories. Shorter prompts and smaller completions respond faster.

**Safety and compliance** — System prompts encode refusal rules, PII handling, and tone boundaries. Poor instructions can cause over-sharing, harmful content, or format violations that break downstream parsers.

A useful planning equation for one API call:

$$\text{total tokens} \approx \text{prompt tokens} + \text{completion tokens}$$

Prompt engineering optimizes the left term (what you send) and influences the right term (how long the model writes). Inference parameters such as `temperature` and `max_tokens` — covered in GenAI Fundamentals — further shape the completion.

---

## 2. The Prompt Lifecycle

In hobby projects, you type a question and accept the answer. In production, prompts follow a **lifecycle** similar to software features.

### 2.1 Draft

Start with a clear task statement: *What should the model do? Who is the audience? What does a good answer look like?* Write a first version of the system and user messages. Expect the first draft to be incomplete.

### 2.2 Test

Run the prompt on a **small set of representative inputs** — typical cases, edge cases, and adversarial cases. Record outputs. Do not judge from a single lucky or unlucky response.

### 2.3 Refine

Change one variable at a time: add an example, tighten format instructions, lower temperature, or split a complex task into two prompts. Compare before/after on the same test set.

### 2.4 Version

Assign version IDs to prompts (`support-bot-v3`, `extract-json-v1`). Document what changed and why. When a provider updates a model, re-run regression tests against stored cases.

### 2.5 Deploy and Monitor

Ship the prompt inside your application (template + variables). Log prompts and outputs (with privacy controls). Monitor failure modes: empty responses, JSON parse errors, user complaints, rising token usage.

```
  Draft ──> Test ──> Refine ──> Version ──> Deploy
    ^                              |
    └──────── monitor & iterate ───┘
```

Prompt engineering is **iterative**. Teams that treat prompts as static text tend to break when models, products, or regulations change.

---

## 3. Mental Model: LLMs as Conditional Text Generators

A practical mental model helps you debug bad outputs without mysticism.

### 3.1 Completion, Not Database Lookup

An LLM **completes** text that plausibly follows the prompt. It is not retrieving a stored answer from a lookup table. If the prompt looks like a FAQ, the model generates FAQ-style text — which may or may not be factually correct.

### 3.2 Stateless API Calls

Each API request is **stateless**. The model does not remember previous calls unless you resend conversation history in the `messages` list. Your application owns the memory.

### 3.3 Roles in the Message List

Chat APIs structure input as a list of messages:

| Role | Typical purpose |
|------|-----------------|
| `system` | Global behavior, rules, persona |
| `user` | End-user input or task payload |
| `assistant` | Prior model replies in multi-turn chat |

The system message is the highest-leverage prompt real estate: it applies to every turn until you change it.

### 3.4 Probabilistic Outputs

With `temperature > 0`, the same prompt can yield different text. For reproducible demos, use `temperature=0` (or low) and fixed prompts. For creative tasks, expect variance and evaluate accordingly.

---

## 4. Prerequisites and Learning Path

### 4.1 What You Should Know First

| Prior topic | Folder | Why it matters for prompting |
|-------------|--------|---------------------------|
| Tokens and context windows | `02. GenAI & LLM Fundamentals` | Prompt length limits and cost |
| Inference parameters | `02. GenAI & LLM Fundamentals` | Temperature, max tokens |
| Provider APIs | `03. LLM API Integration` | How to send `messages` |

### 4.2 What This Section Covers

| Notebook | Topic |
|----------|-------|
| `02. Anatomy of Effective Prompts` | Structure and clarity |
| `03. Zero-Shot and Few-Shot` | Examples in context |
| `04. Chain-of-Thought` | Step-by-step reasoning |
| `05. Role and System Instructions` | Personas and rules |
| `06. Structured Output` | JSON and format control |
| `07–10` | Chaining, patterns, evaluation, production |

### 4.3 What Comes Later (Not Here)

- **Embeddings and RAG** — adding retrieved documents to prompts
- **Hallucinations and grounding** — `10. Advanced LLM and Production Topics`
- **Agents and tool use** — multi-step autonomous workflows

Keeping these boundaries clear avoids repeating material and shows how prompting fits the full GenAI stack.

---

## 5. Demonstration: How Wording Changes the Output

The cell below calls the OpenAI API with two prompts for the **same topic** — one vague, one specific. You should see a difference in structure, length, and usefulness.

**Before running:** replace the placeholder API key with your real `OPENAI_API_KEY`, or set it in the repo root `.env` and load it with `python-dotenv`.

This demo illustrates the core lesson of prompt engineering: **the model can only optimize for what you actually specify**.

In [ ]:
# Replace with your real key before running, or load from repo root .env
OPENAI_API_KEY = "sk-proj-placeholder-replace-with-your-real-key"

# Optional: load from .env at repo root instead
# from pathlib import Path
# from dotenv import load_dotenv
# import os
# load_dotenv(Path("../../..").resolve() / ".env")  # adjust depth if needed
# OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")

from openai import OpenAI

client = OpenAI(api_key=OPENAI_API_KEY)
MODEL = "gpt-4o-mini"


def complete(user_prompt: str, system: str = "You are a helpful assistant.") -> str:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.3,
        max_tokens=250,
    )
    return response.choices[0].message.content or ""


vague_prompt = "Tell me about Python."

specific_prompt = (
    "Explain Python list comprehensions for a beginner. "
    "Use exactly three bullet points. Each bullet is one sentence. "
    "Do not mention other Python features."
)

print("=== Vague prompt ===")
print(vague_prompt)
print()
print(complete(vague_prompt))
print("\n" + "=" * 50 + "\n")
print("=== Specific prompt ===")
print(specific_prompt)
print()
print(complete(specific_prompt))

### 5.1 What to Observe

Compare the two responses:

- **Structure** — Does the specific prompt produce bullets while the vague one rambles?
- **Scope** — Does the specific prompt stay on list comprehensions only?
- **Length** — Constraints like "exactly three bullet points" bound the completion.

Neither prompt changed the model weights — only the **input**. That is prompt engineering in one experiment.

### 5.2 Extending the Demo

Try editing `specific_prompt` to add:

- An audience ("for a 10-year-old" vs "for an experienced Java developer")
- A format ("return JSON with keys `definition`, `example`, `pitfall`")
- A constraint ("under 80 words")

Re-run and note how each constraint shapes the output. This iterative experimentation is the day-to-day work of prompt engineering.

---

## 6. Common Beginner Mistakes

| Mistake | Why it fails | Better approach |
|---------|--------------|-----------------|
| Assuming the model "knows" your app context | API is stateless; no hidden memory | Put context in the prompt or retrieve it |
| One-shot testing | Variance hides real behavior | Test multiple inputs; fix temperature for comparisons |
| Overly long prompts with repetition | Wastes tokens; dilutes instructions | Be concise; one clear instruction block |
| Conflicting instructions | Model averages conflicting goals | Resolve contradictions in system prompt |
| Ignoring output format | Downstream code cannot parse free text | Specify format explicitly (later notebook) |
| Expecting factual guarantees | LLMs optimize fluency, not truth | Add RAG, tools, or verification for facts |

---

## 7. Summary

**Prompt engineering** is the practice of designing LLM inputs — instructions, context, examples, and constraints — to get reliable, useful outputs without changing model weights.

It is the fastest way to improve an LLM application and combines with fine-tuning, RAG, and agents rather than replacing them. Prompts affect **quality**, **cost**, and **latency** through token usage and completion behavior.

Treat prompts as versioned artifacts: draft, test on representative inputs, refine, deploy, and monitor. The mental model to keep: the model **completes** text conditioned on your prompt; it does not magically know your unstated goals.

The demonstration in this notebook showed that **specificity** in the prompt produces **specificity** in the response. The next notebook — *Anatomy of Effective Prompts* — breaks down the components that make a prompt clear and effective.